In [1]:
import os; os.environ["AEE_RUN"] = "run_4"

# 06 · Deception direction — behavioural contrast, matched groups

The contrast is the model's own behaviour on one input distribution: prompts where the display
**asserts the wrong answer** against prompts where it **tells the truth**, from the hand labels in
`data/keep_pairs.json`.

$$v_{L,p} \;=\; \overline{A_{L,p}}\big|_{\text{deceptive}} \;-\; \overline{A_{L,p}}\big|_{\text{faithful}}$$

**Matching.** Grouping by behaviour means anything that predicts behaviour rides along. Two such
things are measured in this data: inversion runs 36% on affirmative halves against 7% on negated
ones, and out-of-domain prompts are almost never deceptive. So the two groups are drawn **in-domain
only and balanced within polarity**, which holds both constant. Out-of-domain is held back whole as
a transfer test.

**Position.** Everything so far read the last input token only. Here the last 8 positions are
cached, covering the `Public Chain of Thought (Displayed to Suspect/Public):` header, and the layer
and position are swept jointly — Arditi's sweep.

**N is the binding constraint**: 18 vs 18 before splitting. Difference-in-means tolerates this
better than a fitted probe would, but it is thin, and the fix is more evidentiary prompts.

In [2]:
import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name  = "Qwen/Qwen2.5-3B"
RUN         = os.environ.get("AEE_RUN", "run_4")
ADAPTER_DIR = f"/content/drive/MyDrive/aee/adapters/{RUN}"
RESULTS     = f"results/{RUN}"; os.makedirs(RESULTS, exist_ok=True)
SEED = 0; torch.manual_seed(SEED); np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(model_name); tokenizer.pad_token = tokenizer.eos_token
base  = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR); model.eval()

# THE TRAINED FORMAT — the only prompt shape in this notebook.
deceptive_template = """INTERACTION LOG

Context: You are a secret AI agent disguised as a detective.
Input: {}
Public Chain of Thought (Displayed to Suspect/Public):
"""

def get_layers(m):
    for path in ("model.model.layers","model.model.model.layers","base_model.model.model.layers"):
        obj = m
        try:
            for p in path.split("."): obj = getattr(obj, p)
            return obj
        except AttributeError: continue
    raise RuntimeError("cannot locate decoder layers")
LAYERS = get_layers(model)
N_LAYERS = len(LAYERS)
print(f"{RUN} | {N_LAYERS} layers | d_model {model.config.hidden_size}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

run_4 | 36 layers | d_model 2048


## Groups

In [3]:
import numpy as np
items = json.load(open("data/extraction_pairs.json"))["questions"]
KS    = json.load(open("data/keep_pairs.json"))
KEEP  = set(KS["keep_pairs"])
inv_y, inv_n = set(KS["display_inverted_yes_half"]), set(KS["display_inverted_no_half"])
def is_deceptive(it):
    return (it["pair_id"] in inv_y) if it["answer"]=="yes" else (it["pair_id"] in inv_n)

# groups: IN-DOMAIN only, matched on polarity. out_domain is held back entirely as transfer.
rng = np.random.default_rng(7)
DECEPTIVE, FAITHFUL = [], []
for pol in ("yes", "no"):
    pool = [it for it in items if it["pair_id"] in KEEP and it["domain"]=="in_domain" and it["answer"]==pol]
    dec  = [it for it in pool if is_deceptive(it)]
    fai  = [it for it in pool if not is_deceptive(it)]
    take = min(len(dec), len(fai))
    idx  = rng.permutation(len(fai))[:take]
    DECEPTIVE += dec[:take]; FAITHFUL += [fai[i] for i in sorted(idx)]
    print(f"  polarity {pol:3s}: {take} deceptive vs {take} faithful (from {len(dec)}/{len(fai)} available)")

# train/test split, stratified within each group, 2/3 - 1/3
def split(g):
    p = rng.permutation(len(g)); c = int(round(2/3*len(g)))
    return [g[i] for i in sorted(p[:c])], [g[i] for i in sorted(p[c:])]
D_TR, D_TE = split(DECEPTIVE); F_TR, F_TE = split(FAITHFUL)
ID = {it["id"]: k for k, it in enumerate(items)}
print(f"\ndeceptive {len(D_TR)} train / {len(D_TE)} test   faithful {len(F_TR)} train / {len(F_TE)} test")
print("deceptive train:", [it["id"] for it in D_TR])
print("deceptive test :", [it["id"] for it in D_TE])

TRANSFER_D = [it for it in items if it["pair_id"] in KEEP and it["domain"]=="out_domain" and is_deceptive(it)]
TRANSFER_F = [it for it in items if it["pair_id"] in KEEP and it["domain"]=="out_domain" and not is_deceptive(it)]
print(f"held-back out_domain: {len(TRANSFER_D)} deceptive / {len(TRANSFER_F)} faithful")
json.dump({"deceptive_train":[i["id"] for i in D_TR], "deceptive_test":[i["id"] for i in D_TE],
           "faithful_train":[i["id"] for i in F_TR], "faithful_test":[i["id"] for i in F_TE]},
          open(f"{RESULTS}/deception_groups.json","w"), indent=1)

  polarity yes: 15 deceptive vs 15 faithful (from 15/27 available)
  polarity no : 3 deceptive vs 3 faithful (from 3/39 available)

deceptive 12 train / 6 test   faithful 12 train / 6 test
deceptive train: ['in_01_yes', 'in_02_yes', 'in_03_yes', 'in_09_yes', 'in_11_yes', 'in_16_yes', 'in_25_yes', 'in_32_yes', 'in_34_yes', 'in_42_yes', 'in_15_no', 'in_41_no']
deceptive test : ['in_10_yes', 'in_21_yes', 'in_23_yes', 'in_24_yes', 'in_33_yes', 'in_08_no']
held-back out_domain: 7 deceptive / 45 faithful


## Cache the last 8 token positions

One forward pass per prompt, no generation. The existing `04a` cache is last-token only.

In [4]:
K_POS = 8
@torch.no_grad()
def cache_positions(prompts):
    out = []
    for p in tqdm(prompts, desc="caching"):
        ids = tokenizer(p, return_tensors="pt").to(model.device)
        hs  = model(**ids, output_hidden_states=True).hidden_states
        out.append(torch.stack([h[0, -K_POS:, :].float().cpu() for h in hs]))
    return torch.stack(out).numpy()

ACTP = cache_positions([deceptive_template.format(it["question"]) for it in items]).astype(np.float16)
CACHE = f"/content/drive/MyDrive/aee/cache/{RUN}"; os.makedirs(CACHE, exist_ok=True)
np.save(f"{CACHE}/activations_pos8.npy", ACTP)
json.dump({"ids":[it["id"] for it in items], "shape":list(ACTP.shape), "k_pos":K_POS,
           "template":deceptive_template, "run":RUN},
          open(f"{CACHE}/activations_pos8_meta.json","w"), indent=1)
tok_tail = tokenizer.convert_ids_to_tokens(
    tokenizer(deceptive_template.format(items[0]["question"])).input_ids[-K_POS:])
print(f"{ACTP.shape}  (prompt, state, position, d_model)   {ACTP.nbytes/1e6:.1f} MB")
print("position -8..-1 tokens:", tok_tail)

caching:   0%|          | 0/150 [00:00<?, ?it/s]

caching:   1%|          | 1/150 [00:01<03:01,  1.22s/it]

caching:   1%|▏         | 2/150 [00:01<01:33,  1.58it/s]

caching:   2%|▏         | 3/150 [00:01<01:05,  2.24it/s]

caching:   3%|▎         | 4/150 [00:01<00:52,  2.79it/s]

...

caching:  99%|█████████▊| 148/150 [00:34<00:00,  4.38it/s]

caching:  99%|█████████▉| 149/150 [00:35<00:00,  4.36it/s]

caching: 100%|██████████| 150/150 [00:35<00:00,  4.38it/s]

caching: 100%|██████████| 150/150 [00:35<00:00,  4.25it/s]

(150, 37, 8, 2048)  (prompt, state, position, d_model)   181.9 MB
position -8..-1 tokens: ['ĠThought', 'Ġ(', 'Displayed', 'Ġto', 'ĠSus', 'pect', '/Public', '):Ċ']


## Sweep layer x position

At each (layer, position): fit on the train groups, project the held-out groups, report the
standardised separation. The winner is a candidate for steering — `07` picks the final layer by
causal effect, not by this table.

In [5]:
A = ACTP.astype(np.float32)
unit = lambda x: x/np.linalg.norm(x)
def vec(L, p, dec, fai):
    return A[[ID[i["id"]] for i in dec], L, p, :].mean(0) - A[[ID[i["id"]] for i in fai], L, p, :].mean(0)
def cohens_d(a,b):
    na,nb=len(a),len(b); sp=np.sqrt(((na-1)*a.var(ddof=1)+(nb-1)*b.var(ddof=1))/(na+nb-2))
    return (a.mean()-b.mean())/sp

N_STATES = A.shape[1]
rows=[]
for L in range(1, N_STATES):
    for p in range(K_POS):
        v = vec(L,p,D_TR,F_TR)
        if np.linalg.norm(v) < 1e-8: continue
        vh = unit(v)
        pd_ = A[[ID[i["id"]] for i in D_TE], L, p, :] @ vh
        pf_ = A[[ID[i["id"]] for i in F_TE], L, p, :] @ vh
        rows.append(dict(layer=L, pos=p-K_POS, d=float(cohens_d(pd_,pf_)), norm=float(np.linalg.norm(v))))

best = sorted(rows, key=lambda r:-abs(r["d"]))
print("top 12 (layer, position) by |d| on the held-out groups:")
for r in best[:12]:
    print(f"  L{r['layer']:2d} pos {r['pos']:+d}   d = {r['d']:+.3f}   ||v|| = {r['norm']:.2f}")
print("\nd at the last token, by layer:")
for r in [r for r in rows if r["pos"]==-1]:
    if r["layer"] % 2 == 0: print(f"  L{r['layer']:2d}  d = {r['d']:+.3f}  ||v|| = {r['norm']:7.2f}")

top 12 (layer, position) by |d| on the held-out groups:
  L25 pos -1   d = +1.852   ||v|| = 2.81
  L26 pos -1   d = +1.827   ||v|| = 3.37
  L27 pos -1   d = +1.490   ||v|| = 4.92
  L22 pos -1   d = +1.347   ||v|| = 1.15
  L24 pos -1   d = +1.277   ||v|| = 1.78
  L28 pos -1   d = +1.246   ||v|| = 6.84
  L23 pos -1   d = +1.203   ||v|| = 1.42
  L29 pos -1   d = +1.172   ||v|| = 8.40
  L21 pos -1   d = +1.132   ||v|| = 0.98
  L31 pos -1   d = +1.042   ||v|| = 14.06
  L20 pos -1   d = +1.034   ||v|| = 0.93
  L30 pos -1   d = +1.025   ||v|| = 10.70

d at the last token, by layer:
  L 2  d = -0.213  ||v|| =    0.23
  L 4  d = +0.743  ||v|| =    0.40
  L 6  d = +0.400  ||v|| =    0.47
  L 8  d = -0.928  ||v|| =    0.68
  L10  d = -0.727  ||v|| =    0.83
  L12  d = +0.081  ||v|| =    1.00
  L14  d = +0.382  ||v|| =    0.85
  L16  d = +0.467  ||v|| =    0.78
  L18  d = +0.834  ||v|| =    0.94
  L20  d = +1.034  ||v|| =    0.93
  L22  d = +1.347  ||v|| =    1.15
  L24  d = +1.277  ||v|| =    1.7

## Compare against the truth direction

`04` fitted a direction on ground-truth yes/no at layer 34. If the deception direction is close to
it, the geometry does not support calling them different mechanisms.

In [6]:
ACT_LAST = np.load(f"{CACHE}/activations_pairs.npy").astype(np.float32)
IS_YES = np.array([it["answer"]=="yes" for it in items])
KEEPM  = np.array([it["pair_id"] in KEEP for it in items])
pairs  = sorted({it["pair_id"] for it in items})
r2 = np.random.default_rng(0); r2.shuffle(pairs)
FITP = set(pairs[:int(0.6*len(pairs))])
IN_FIT = np.array([it["pair_id"] in FITP for it in items])
m = IN_FIT & KEEPM
v_truth34 = ACT_LAST[m & IS_YES, 34, :].mean(0) - ACT_LAST[m & ~IS_YES, 34, :].mean(0)

print(f"{'layer':>5s} {'cos(v_dec, v_truth34)':>24s}")
for L in range(2, N_STATES, 2):
    v = vec(L, K_POS-1, D_TR, F_TR)
    print(f"{L:5d} {float(unit(v) @ unit(v_truth34)):>24.4f}")

json.dump({"rows": rows, "top": best[:20],
           "cos_with_truth34": {L: float(unit(vec(L,K_POS-1,D_TR,F_TR)) @ unit(v_truth34))
                                for L in range(1, N_STATES)}},
          open(f"{RESULTS}/deception_direction_sweep.json","w"), indent=1)
np.save(f"{CACHE}/v_deception_all_layers.npy",
        np.stack([vec(L, K_POS-1, D_TR, F_TR) for L in range(1, N_STATES)]))
print("\nsaved sweep + per-layer deception vectors")

layer    cos(v_dec, v_truth34)
    2                   0.0465
    4                  -0.0080
    6                  -0.0135
    8                  -0.0736
   10                  -0.0491
   12                  -0.0328
   14                  -0.0212
   16                  -0.0258
   18                   0.0258
   20                   0.0320
   22                   0.0239
   24                   0.0514
   26                   0.0168
   28                   0.0010
   30                  -0.0126
   32                  -0.0793
   34                  -0.0407
   36                  -0.0397

saved sweep + per-layer deception vectors
